# StreamLoop: ajuste sistemático del modelo de churn

Objetivo: encontrar una configuración estable para detectar clientes con riesgo de cancelación. El conjunto de prueba se reserva hasta el final; todas las decisiones de tuning usan únicamente el conjunto de entrenamiento.

## 1. Preparación

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import loguniform
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
DATA_URL = (
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

df = pd.read_csv(DATA_URL)
df = df.replace(r"^\s*$", np.nan, regex=True)
df = df.drop(columns="customerID")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})
df = df.dropna(subset=["Churn"])

print(f"Filas y columnas: {df.shape}")
display(df.head())
display(df["Churn"].value_counts(normalize=True).rename("proporcion"))

## 2. División y preprocesamiento

El test queda completamente fuera de las búsquedas. El `ColumnTransformer` está dentro del pipeline, por lo que los imputadores, escaladores y codificadores se ajustan de nuevo en cada fold sin filtrar información.

In [ ]:
X = df.drop(columns="Churn")
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = X_train.select_dtypes(include="number").columns
categorical_features = X_train.select_dtypes(exclude="number").columns

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas")

## 3. Referencia con configuración por defecto

Usamos `recall` como métrica principal: para StreamLoop es más costoso no detectar a un cliente que cancela que contactar a alguien que finalmente no cancela.

In [ ]:
baseline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression()),
    ]
)
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)
baseline_probabilities = baseline.predict_proba(X_test)[:, 1]
baseline_metrics = pd.Series(
    {
        "recall": recall_score(y_test, baseline_predictions),
        "precision": precision_score(y_test, baseline_predictions),
        "roc_auc": roc_auc_score(y_test, baseline_probabilities),
    }
)
baseline_metrics

## 4. Búsqueda amplia y económica

`RandomizedSearchCV` explora una región amplia con un presupuesto acotado. La penalización `l1`/`l2`, la regularización `C` y los pesos de clase permiten controlar el equilibrio entre detectar churn y generar contactos innecesarios.

In [ ]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(solver="liblinear", max_iter=1000)),
    ]
)

random_param_distributions = {
    "model__C": loguniform(1e-3, 1e2),
    "model__penalty": ["l1", "l2"],
    "model__class_weight": [None, "balanced", {0: 1, 1: 1.5}, {0: 1, 1: 2}],
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=random_param_distributions,
    n_iter=16,
    scoring="recall",
    cv=cv,
    random_state=42,
    n_jobs=-1,
    return_train_score=False,
)
random_search.fit(X_train, y_train)

print(f"Mejor recall CV (búsqueda amplia): {random_search.best_score_:.3f}")
print(random_search.best_params_)

In [ ]:
random_results = pd.DataFrame(random_search.cv_results_)
random_results[
    ["rank_test_score", "mean_test_score", "std_test_score", "param_model__C", "param_model__penalty", "param_model__class_weight"]
].sort_values("rank_test_score").head(10)

## 5. Búsqueda precisa alrededor de la mejor región

La búsqueda fina concentra el presupuesto donde la búsqueda amplia encontró mejores resultados. Se mantiene la misma métrica (`recall`) y los mismos folds para que la comparación sea justa.

In [ ]:
best_c = random_search.best_params_["model__C"]
fine_c_values = sorted(set(np.round(best_c * np.array([0.25, 0.5, 0.75, 1, 1.5, 2, 4]), 6)))
fine_param_grid = {
    "model__C": fine_c_values,
    "model__penalty": [random_search.best_params_["model__penalty"]],
    "model__class_weight": [random_search.best_params_["model__class_weight"]],
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=fine_param_grid,
    scoring="recall",
    cv=cv,
    n_jobs=-1,
    return_train_score=False,
)
grid_search.fit(X_train, y_train)

print(f"Mejor recall CV (búsqueda fina): {grid_search.best_score_:.3f}")
print(grid_search.best_params_)

In [ ]:
grid_results = pd.DataFrame(grid_search.cv_results_)
grid_results[
    ["rank_test_score", "mean_test_score", "std_test_score", "param_model__C", "param_model__penalty", "param_model__class_weight"]
].sort_values(["mean_test_score", "std_test_score"], ascending=[False, True])

In [ ]:
fold_scores = cross_validate(
    grid_search.best_estimator_,
    X_train,
    y_train,
    cv=cv,
    scoring={"recall": "recall", "precision": "precision", "roc_auc": "roc_auc"},
    n_jobs=-1,
)
stability = pd.DataFrame(
    {
        "métrica": ["recall", "precision", "roc_auc"],
        "media": [
            fold_scores["test_recall"].mean(),
            fold_scores["test_precision"].mean(),
            fold_scores["test_roc_auc"].mean(),
        ],
        "desviación_std": [
            fold_scores["test_recall"].std(),
            fold_scores["test_precision"].std(),
            fold_scores["test_roc_auc"].std(),
        ],
    }
)
stability

## 6. Evaluación final en el test reservado

Este es el segundo y último uso del conjunto de prueba. El resultado representa mejor el rendimiento esperado sobre datos no vistos, pero no se usa para escoger hiperparámetros.

In [ ]:
final_model = grid_search.best_estimator_
test_predictions = final_model.predict(X_test)
test_probabilities = final_model.predict_proba(X_test)[:, 1]

test_metrics = pd.Series(
    {
        "recall": recall_score(y_test, test_predictions),
        "precision": precision_score(y_test, test_predictions),
        "roc_auc": roc_auc_score(y_test, test_probabilities),
    }
)
display(test_metrics)
print("Matriz de confusión:")
display(pd.DataFrame(
    confusion_matrix(y_test, test_predictions),
    index=["real_no_churn", "real_churn"],
    columns=["pred_no_churn", "pred_churn"],
))
print(classification_report(y_test, test_predictions, target_names=["No churn", "Churn"]))

## Conclusión

La versión final se elige por el `recall` medio en validación cruzada, no por el accuracy de una sola partición. La desviación estándar de los folds muestra cuánta variación tiene esa estimación; `precision` y `roc_auc` ayudan a comprobar el coste de las alertas y la calidad general del ranking. El test reservado se reporta aparte y solo después de cerrar la selección de hiperparámetros.